# 임베딩 사전 작업 노트북
**파싱 → 청킹 → 임베딩 → 결과 저장 & 공유**

팀원과 공유할 인덱스 파일(`.pkl`)  생성  
생성된 파일을 Drive에 올리면 검색/생성 단계는 API 없이 바로 시작 가능

| 단계 | 내용 | 출력 |
|------|------|------|
| Phase 1 | PDF 파싱 (PyMuPDF) | 파일별 페이지 수 / 문자 수 |
| Phase 2 | 청킹 (sliding window 800자) | 청크 수 / 크기 분포 |
| Phase 3 | 임베딩 (Upstage embedding-passage) | 벡터 shape / 소요시간 |
| 저장 | `index.pkl` 저장 → Drive 공유 | 파일 크기 |

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. 작업 디렉토리 설정 및 패키지 설치

In [ ]:
import os

REPO_PATH = '/content/drive/MyDrive/gragra'
os.chdir(REPO_PATH)
print(f"작업 디렉토리: {os.getcwd()}")

작업 디렉토리: /content/drive/MyDrive/gragra


In [ ]:
!pip install pymupdf -q       # fitz (PyMuPDF)
!pip install rank-bm25 -q     # BM25 인덱스 (검색 단계용, 미리 포함)
print("설치 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 62.8 MB/s eta 0:00:00
설치 완료


## 3. API 키 설정

왼쪽 사이드바 **🔑 Secrets 탭**에서 키를 등록 필요

| Secret 이름 | 설명 | 등록 시점 |
|---|---|---|
| `UPSTAGE_API_KEY` | Solar LLM 호출 | 개인키 발급|
| `HACKATHON_KEY` | 테스트 셋 복호화 | 대회 당일 |

> Secrets 탭에서 키를 추가한 뒤, **"Notebook access" 토글을 ON**으로 설정 필수

In [ ]:
import os
from google.colab import userdata

# os.environ["UPSTAGE_API_KEY"] = userdata.get("UPSTAGE_API_KEY")

os.environ["HACKATHON_KEY"] = userdata.get("HACKATHON_KEY")

print("UPSTAGE_API_KEY 로드 완료:", "OK" if os.environ.get("UPSTAGE_API_KEY") else "❌ 키 없음")

UPSTAGE_API_KEY 로드 완료: ❌ 키 없음


## 4. Import 및 설정

In [ ]:
from __future__ import annotations

import io, os, re, sys, time, pickle, json
import urllib.request, urllib.error
from pathlib import Path

import fitz
import numpy as np

if isinstance(sys.stdout, io.TextIOWrapper):
    sys.stdout.reconfigure(encoding="utf-8")

# ── 설정 ──────────────────────────────────────────────────────────────────
CORPUS_DIR    = "/content/drive/MyDrive/gragra/2026-up-tech-data/corpus"
INDEX_PATH    = "index/index_v1.pkl"
CHUNK_SIZE    = 800
CHUNK_OVERLAP = 150

# ── Embedder (upstage_client.py 인라인) ───────────────────────────────────
EMBED_URL            = "https://api.upstage.ai/v1/embeddings"
EMBED_PASSAGE_MODEL  = "embedding-passage"
CACHE_DIR            = Path(".cache")

_RETRY_STATUS = {429, 500, 502, 503, 504}
_MAX_RETRIES  = 6


def _api_key() -> str:
    key = os.environ.get("UPSTAGE_API_KEY")
    if not key:
        raise EnvironmentError("UPSTAGE_API_KEY 환경변수가 없습니다.")
    return key


def _retry_after(e: urllib.error.HTTPError) -> int | None:
    try:
        v = e.headers.get("Retry-After")
        return int(v) if v else None
    except (TypeError, ValueError):
        return None


def _urlopen_with_retry(req: urllib.request.Request, timeout: int, label: str) -> dict:
    backoff = 5
    for attempt in range(1, _MAX_RETRIES + 1):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return json.loads(resp.read().decode("utf-8"))
        except urllib.error.HTTPError as e:
            body = e.read().decode("utf-8", errors="replace")
            if e.code not in _RETRY_STATUS or attempt == _MAX_RETRIES:
                raise RuntimeError(f"{label} 실패 [{e.code}]: {body}") from e
            wait_s = _retry_after(e) or backoff
            print(f"  [{label}] HTTP {e.code} ({attempt}/{_MAX_RETRIES}) — {wait_s}s 후 재시도")
            time.sleep(wait_s)
            backoff = min(backoff * 2, 120)
    raise RuntimeError(f"{label} 실패 (재시도 소진)")


class Embedder:
    """Upstage embedding-passage API + 배치 디스크 캐시."""

    def __init__(self, cache_dir: Path = CACHE_DIR, batch_size: int = 100) -> None:
        self.cache_dir  = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.batch_size = batch_size

    def embed_passages(self, texts: list[str], cache_key: str) -> np.ndarray:
        emb_path = self.cache_dir / f"emb_{cache_key}.npy"
        if emb_path.exists():
            print(f"  [embed-cache] {emb_path.name}")
            return np.load(emb_path)

        n_batches = (len(texts) + self.batch_size - 1) // self.batch_size
        print(f"  [embed] {len(texts)}개 → {n_batches}배치")

        batch_dir = self.cache_dir / f"emb_{cache_key}_batches"
        batch_dir.mkdir(parents=True, exist_ok=True)

        vecs: list[np.ndarray] = []
        for bi in range(n_batches):
            batch_path = batch_dir / f"{bi:05d}.npy"
            if batch_path.exists():
                v = np.load(batch_path)
                print(f"    배치 {bi+1}/{n_batches}: 캐시")
            else:
                batch = texts[bi * self.batch_size : (bi + 1) * self.batch_size]
                t0 = time.time()
                v   = self._embed_batch(batch)
                np.save(batch_path, v)
                print(f"    배치 {bi+1}/{n_batches}: {len(batch)}개 → {time.time()-t0:.1f}s")
            vecs.append(v)

        out = np.vstack(vecs).astype(np.float32)
        np.save(emb_path, out)
        print(f"  [embed] 저장: {emb_path.name} {out.shape}")
        return out

    def _embed_batch(self, texts: list[str]) -> np.ndarray:
        payload = {"model": EMBED_PASSAGE_MODEL, "input": texts}
        req = urllib.request.Request(
            url     = EMBED_URL,
            data    = json.dumps(payload).encode("utf-8"),
            headers = {
                "Authorization": f"Bearer {_api_key()}",
                "Content-Type":  "application/json",
            },
        )
        data = _urlopen_with_retry(req, timeout=120, label=f"Embedding")
        return np.array([d["embedding"] for d in data["data"]], dtype=np.float32)


print("설정 완료")
print(f"  corpus : {Path(CORPUS_DIR).resolve()}")
print(f"  index  : {INDEX_PATH}")

설정 완료
  corpus : /content/drive/MyDrive/gragra/2026-up-tech-data/corpus
  index  : index/index_v1.pkl


## Phase 1 — PDF 파싱

PyMuPDF(`fitz`)로 각 PDF에서 텍스트를 추출
파일별 페이지 수 / 추출 문자 수를 출력하여 파싱 품질을 확인

In [ ]:
def _parse_pdf(path: Path) -> tuple[str, int]:
    """PDF 텍스트 추출. (전체 텍스트, 페이지 수) 반환."""
    doc   = fitz.open(str(path))
    pages = [page.get_text() for page in doc]
    doc.close()
    text  = "\n\n".join(p for p in pages if p.strip())
    return text, len(pages)


def _chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """문단 경계 우선 sliding window 청킹."""
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    if not text:
        return []
    chunks, i, n = [], 0, len(text)
    while i < n:
        end = min(i + size, n)
        if end < n:
            nl = text.rfind("\n\n", i, end)
            if nl > i + size // 2:
                end = nl
        chunk = text[i:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= n:
            break
        i = end - overlap if end - overlap > i else end
    return chunks


print("함수 정의 완료")

함수 정의 완료


In [ ]:
# ── PDF 파싱 실행 ──────────────────────────────────────────────────────────
corpus  = Path(CORPUS_DIR)
pdfs    = sorted(corpus.glob("**/*.pdf"))
print(f"PDF 파일: {len(pdfs)}개\n")
print(f"{'파일명':<35} {'페이지':>5} {'문자 수':>9} {'미리보기'}")
print("─" * 85)

raw_texts: list[tuple[str, str]] = []   # (source, text)
t0 = time.time()

for pdf in pdfs:
    text, n_pages = _parse_pdf(pdf)
    raw_texts.append((pdf.name, text))
    preview = text[:60].replace("\n", " ")
    print(f"{pdf.name:<35} {n_pages:>5} {len(text):>9,}자  {preview}...")

elapsed = time.time() - t0
print("─" * 85)
total_chars = sum(len(t) for _, t in raw_texts)
print(f"합계: {len(pdfs)}개 PDF  |  {total_chars:,}자  |  {elapsed:.1f}초")

PDF 파일: 150개

파일명                                   페이지      문자 수 미리보기
─────────────────────────────────────────────────────────────────────────────────────
emails_allen-p.pdf                     40   112,039자  ENRON CORPORATION Internal Email Archive  |  Mailbox: allen-...
emails_arnold-j.pdf                    79   253,525자  ENRON CORPORATION Internal Email Archive  |  Mailbox: arnold...
emails_arora-h.pdf                     20    54,721자  ENRON CORPORATION Internal Email Archive  |  Mailbox: arora-...
emails_badeer-r.pdf                    23    76,973자  ENRON CORPORATION Internal Email Archive  |  Mailbox: badeer...
emails_bailey-s.pdf                    32    78,735자  ENRON CORPORATION Internal Email Archive  |  Mailbox: bailey...
emails_bass-e.pdf                      64   181,092자  ENRON CORPORATION Internal Email Archive  |  Mailbox: bass-e...
emails_baughman-d.pdf                  38   109,790자  ENRON CORPORATION Internal Email Archive  |  Mailbox: baughm...
emails_beck-s.pdf

## Phase 2 — 청킹

800자 sliding window(overlap 150자)로 텍스트를 분할

청크 수 / 평균 크기 / 분포를 확인

In [ ]:
# ── 청킹 실행 ─────────────────────────────────────────────────────────────
all_chunks:  list[str] = []
all_sources: list[str] = []

for source, text in raw_texts:
    chunks = _chunk_text(text)
    all_chunks.extend(chunks)
    all_sources.extend([source] * len(chunks))

sizes = [len(c) for c in all_chunks]

print(f"총 청크 수  : {len(all_chunks):,}개")
print(f"평균 크기   : {sum(sizes)/len(sizes):.0f}자")
print(f"최소 / 최대 : {min(sizes)}자 / {max(sizes)}자")
print()

# 크기 구간 분포
bins = [(0,200),(200,400),(400,600),(600,800),(800,1200)]
print(f"{'크기 구간':<15} {'청크 수':>8} {'비율':>8}")
print("─" * 35)
for lo, hi in bins:
    cnt = sum(1 for s in sizes if lo <= s < hi)
    bar = "█" * int(cnt / len(sizes) * 30)
    print(f"{lo:>4}~{hi:<7}자  {cnt:>8,}  {cnt/len(sizes)*100:>6.1f}%  {bar}")

print()
print("샘플 청크 (첫 번째):")
print("─" * 60)
print(all_chunks[0][:300], "...")

총 청크 수  : 46,803개
평균 크기   : 774자
최소 / 최대 : 153자 / 800자

크기 구간               청크 수       비율
───────────────────────────────────
   0~200    자        13     0.0%  
 200~400    자        39     0.1%  
 400~600    자     2,957     6.3%  █
 600~800    자    14,084    30.1%  █████████
 800~1200   자    29,710    63.5%  ███████████████████

샘플 청크 (첫 번째):
────────────────────────────────────────────────────────────
ENRON CORPORATION
Internal Email Archive  |  Mailbox: allen-p  |  47 message(s)
p. 1
CONFIDENTIAL
CONFIDENTIAL - Enron Corporation Internal Records - Reconstructed for Research Purposes
Message 1 of 47
nan
Sender
k..allen@enron.com
Recipients
['pallen70@hotmail.com']
File
allen-p/sent_items/28.
---- ...


## Phase 3 — 임베딩

Upstage `embedding-passage` API로 청크를 벡터화  
배치(100개)별로 캐시 저장

In [ ]:
# ── 임베딩 실행 ────────────────────────────────────────────────────────────
embedder  = Embedder()
cache_key = f"{len(all_chunks)}_{CHUNK_SIZE}_{CHUNK_OVERLAP}_{Path(CORPUS_DIR).name}"

from google.colab import userdata
os.environ["UPSTAGE_API_KEY"] = userdata.get("UPSTAGE_API_KEY")

print(f"cache_key: {cache_key}")
print(f"청크 수  : {len(all_chunks):,}개")
print(f"예상 배치: {(len(all_chunks) + 99) // 100}회\n")

t0  = time.time()
emb = embedder.embed_passages(all_chunks, cache_key=cache_key)
emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12)
elapsed = time.time() - t0

print(f"\n임베딩 완료!")
print(f"  shape    : {emb.shape}  (청크 수 × 벡터 차원)")
print(f"  dtype    : {emb.dtype}")
print(f"  소요시간 : {elapsed:.1f}초")
print(f"  메모리   : {emb.nbytes / 1024**2:.1f} MB")

# 정규화 확인 (norm ≈ 1.0이어야 함)
norms = np.linalg.norm(emb[:5], axis=1)
print(f"\n정규화 확인 (첫 5개 norm): {norms.round(4)}")

cache_key: 46803_800_150_corpus
청크 수  : 46,803개
예상 배치: 469회

  [embed] 46803개 → 469배치
    배치 1/469: 100개 → 2.7s
    배치 2/469: 100개 → 2.6s
    배치 3/469: 100개 → 2.6s
    배치 4/469: 100개 → 2.7s
    배치 5/469: 100개 → 3.2s
    배치 6/469: 100개 → 5.0s
    배치 7/469: 100개 → 2.9s
    배치 8/469: 100개 → 2.5s
    배치 9/469: 100개 → 2.6s
    배치 10/469: 100개 → 3.6s
    배치 11/469: 100개 → 2.7s
    배치 12/469: 100개 → 3.0s
    배치 13/469: 100개 → 3.1s
    배치 14/469: 100개 → 6.7s
    배치 15/469: 100개 → 2.7s
    배치 16/469: 100개 → 2.6s
    배치 17/469: 100개 → 2.6s
    배치 18/469: 100개 → 3.1s
    배치 19/469: 100개 → 2.9s
    배치 20/469: 100개 → 2.7s
    배치 21/469: 100개 → 2.7s
    배치 22/469: 100개 → 2.7s
    배치 23/469: 100개 → 2.6s
    배치 24/469: 100개 → 2.9s
    배치 25/469: 100개 → 2.6s
    배치 26/469: 100개 → 2.6s
    배치 27/469: 100개 → 2.6s
    배치 28/469: 100개 → 2.6s
    배치 29/469: 100개 → 2.9s
    배치 30/469: 100개 → 2.6s
    배치 31/469: 100개 → 2.9s
    배치 32/469: 100개 → 3.1s
    배치 33/469: 100개 → 2.7s
    배치 34/469: 100개 → 3.0s
    배

## 결과 저장

`index.pkl`에 청크·소스·임베딩을 모두 저장  

파싱/임베딩 없이 바로 검색/생성 단계를 실행 가능

In [ ]:
# ── BM25 인덱스 추가 (검색 단계에서 바로 쓸 수 있도록 함께 저장) ──────────
from rank_bm25 import BM25Okapi
print("BM25 인덱스 구축 중...")
bm25 = BM25Okapi([c.lower().split() for c in all_chunks])
print(f"BM25 완료  : {len(all_chunks):,}개 문서\n")

# ── 저장 ──────────────────────────────────────────────────────────────────
index = {
    "chunks":     all_chunks,    # list[str]
    "sources":    all_sources,   # list[str]
    "embeddings": emb,           # np.ndarray (N, D)
    "bm25":       bm25,          # BM25Okapi
    "meta": {
        "n_chunks":   len(all_chunks),
        "emb_shape":  list(emb.shape),
        "chunk_size":  CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "corpus_dir":  CORPUS_DIR,
    }
}

Path(INDEX_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(INDEX_PATH, "wb") as f:
    pickle.dump(index, f)

size_mb = Path(INDEX_PATH).stat().st_size / 1024 / 1024
print(f"저장 완료: {INDEX_PATH}")
print(f"  파일 크기: {size_mb:.1f} MB")
print(f"  청크 수  : {index['meta']['n_chunks']:,}개")
print(f"  임베딩   : {index['meta']['emb_shape']}")
print(f"\n팀원 로드 방법:")
print(f"  import pickle")
print(f"  with open('{INDEX_PATH}', 'rb') as f: index = pickle.load(f)")

BM25 인덱스 구축 중...
BM25 완료  : 46,803개 문서

저장 완료: index/index_v1.pkl
  파일 크기: 814.7 MB
  청크 수  : 46,803개
  임베딩   : [46803, 4096]

팀원 로드 방법:
  import pickle
  with open('index/index_v1.pkl', 'rb') as f: index = pickle.load(f)


## 다운로드 (Colab → 로컬 / Drive 업로드)

In [ ]:
# # 방법 1: 로컬로 다운로드
from google.colab import files
files.download(INDEX_PATH)
print(f"{INDEX_PATH} 다운로드 시작")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

index/index_v1.pkl 다운로드 시작


In [ ]:
# 방법 2: Google Drive에 바로 복사 (팀원 공유용)
import shutil

DRIVE_DEST = "/content/drive/MyDrive/gragra/index_v1.pkl"
Path(DRIVE_DEST).parent.mkdir(parents=True, exist_ok=True)
shutil.copy(INDEX_PATH, DRIVE_DEST)
print(f"Drive 저장 완료: {DRIVE_DEST}")